# Phase 2 — Data Cleaning

This notebook creates the primary modeling table without changing the raw CSV. Phase 2 performs only deterministic corrections: removing structural columns and duplicates, converting impossible zero-seat values to missing, and auditing extreme odometer records.

Statistical imputation and outlier thresholds are deliberately deferred until after train/validation/test splitting to prevent leakage.

## 1. Imports and project paths

In [ ]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.clean_data import build_odometer_review, clean_dataframe, run_cleaning
from src.config import (
    CLEANING_SUMMARY_PATH, ODOMETER_REVIEW_PATH,
    PROCESSED_DATA_PATH, RAW_DATA_PATH,
)
from src.validate_data import file_sha256, load_raw_data

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data: {RAW_DATA_PATH}")
print(f"Cleaned data: {PROCESSED_DATA_PATH}")

## 2. Load and identify the untouched raw file

The SHA-256 hash uniquely identifies the input file. It will be checked again after cleaning to prove that the raw CSV was not overwritten.

In [ ]:
raw_df = load_raw_data(RAW_DATA_PATH)
raw_hash_before = file_sha256(RAW_DATA_PATH)

print(f"Raw shape: {raw_df.shape}")
print(f"Raw SHA-256: {raw_hash_before}")
display(raw_df.head())

## 3. Confirm the structural corrections

`Unnamed: 0` is an old exported index. Once it is excluded, 167 duplicate listings become visible. `car_name` is also redundant because every value is exactly `brand + model`.

In [ ]:
without_export_index = raw_df.drop(columns=["Unnamed: 0"])
duplicate_count = int(without_export_index.duplicated().sum())
expected_car_name = (
    raw_df["brand"].astype(str).str.strip()
    + " "
    + raw_df["model"].astype(str).str.strip()
)
car_name_match_count = int(raw_df["car_name"].eq(expected_car_name).sum())

print(f"Duplicate listings after excluding index: {duplicate_count}")
print(f"car_name equals brand + model: {car_name_match_count:,}/{len(raw_df):,}")

## 4. Investigate the invalid seat values

Both affected models normally appear with five seats in this dataset. However, using the full dataset to learn an imputation rule before splitting would violate the project leakage rules. Phase 2 therefore converts zero to missing and leaves imputation to a training-only pipeline.

In [ ]:
zero_seat_rows = raw_df.loc[raw_df["seats"].eq(0)]
display(zero_seat_rows)

for car_name in zero_seat_rows["car_name"].unique():
    counts = (
        raw_df.loc[raw_df["car_name"].eq(car_name), "seats"]
        .value_counts(dropna=False)
        .sort_index()
    )
    print(f"\nSeat counts for {car_name}:")
    display(counts.rename("row_count").to_frame())

## 5. Audit extreme odometer records

Twelve records exceed 500,000 km. Exact-specification peers make the values look suspicious, and five records have kilometres exactly equal to selling price. The available data still cannot prove the correct replacement. The audit report records the evidence while keeping every value unchanged.

In [ ]:
odometer_review = build_odometer_review(raw_df)

print(f"Rows reviewed: {len(odometer_review)}")
print(
    "Rows where km_driven equals selling_price: "
    f"{int(odometer_review['km_equals_selling_price'].sum())}"
)
display(odometer_review)

## 6. Apply deterministic cleaning in memory

The reusable function applies the same validated steps every time. It returns both the cleaned DataFrame and an audit dictionary.

In [ ]:
cleaned_df, cleaning_audit = clean_dataframe(raw_df)

before_after = pd.DataFrame({
    "metric": ["Rows", "Columns", "Duplicate rows", "Zero-seat rows", "Missing seats"],
    "before": [
        len(raw_df),
        raw_df.shape[1],
        duplicate_count,
        int(raw_df["seats"].eq(0).sum()),
        int(raw_df["seats"].isna().sum()),
    ],
    "after": [
        len(cleaned_df),
        cleaned_df.shape[1],
        int(cleaned_df.duplicated().sum()),
        int(cleaned_df["seats"].eq(0).sum()),
        int(cleaned_df["seats"].isna().sum()),
    ],
})

display(before_after)
display(cleaned_df.head())
print(cleaning_audit)

## 7. Save the Phase 2 outputs

This writes the cleaned CSV, the machine-readable cleaning summary, and the separate odometer-review report. The raw file is not modified.

In [ ]:
cleaning_summary = run_cleaning()

print(f"Cleaned data saved to: {PROCESSED_DATA_PATH}")
print(f"Cleaning summary saved to: {CLEANING_SUMMARY_PATH}")
print(f"Odometer review saved to: {ODOMETER_REVIEW_PATH}")

## 8. Final verification

These assertions make the notebook stop immediately if any Phase 2 guarantee is broken.

In [ ]:
saved_cleaned_df = pd.read_csv(PROCESSED_DATA_PATH)
raw_hash_after = file_sha256(RAW_DATA_PATH)

assert raw_hash_after == raw_hash_before
assert saved_cleaned_df.shape == (15_244, 12)
assert "Unnamed: 0" not in saved_cleaned_df.columns
assert "car_name" not in saved_cleaned_df.columns
assert saved_cleaned_df.duplicated().sum() == 0
assert saved_cleaned_df["seats"].eq(0).sum() == 0
assert saved_cleaned_df["seats"].isna().sum() == 2
assert saved_cleaned_df.drop(columns=["seats"]).isna().sum().sum() == 0
assert saved_cleaned_df["km_driven"].gt(500_000).sum() == 12
assert saved_cleaned_df["selling_price"].gt(0).all()

print("====================================")
print("PHASE 2 CLEANING VALIDATION PASSED")
print("====================================")
print(f"Cleaned shape: {saved_cleaned_df.shape}")
print(f"Raw file unchanged: {raw_hash_after == raw_hash_before}")

## Phase 2 conclusion

The primary modeling table now has 15,244 rows and 12 columns. Structural leakage risks have been removed, the raw source is unchanged, and uncertain values have not been guessed. The two seat values remain missing for training-only imputation, while the 12 suspicious odometer values remain available for Phase 3 analysis and later sensitivity testing.